In [ ]:
import numpy as np

from vtkmodules.util.numpy_support import numpy_to_vtk
from vtkmodules.util.data_model import (
    UnstructuredGrid,
    Points,
    CellArray
)

from vtkmodules.vtkCommonDataModel import (
    VTK_QUAD
)

from vtkmodules.vtkRenderingCore import (
    vtkActor,
    vtkDataSetMapper,
    vtkRenderer,
    vtkRenderWindow
)

from trame.app import get_server
from trame.ui.vuetify3 import VAppLayout
from trame.widgets import vtk as vtk_widgets


In [ ]:

coordinates = np.array([
    [0.0, 0.0, 0.0],  # point 0
    [1.0, 0.0, 0.0],  # point 1
    [1.0, 1.0, 0.0],  # point 2
    [0.0, 1.0, 0.0],  # point 3
], dtype=np.float64)

points = Points(data=coordinates)

# ============================================================
# Create vtkCellArray
# ============================================================

connectivity = np.array([
    [0, 1, 2, 3],
], dtype=np.int64)

connectivity_flat = connectivity.ravel()

cell_types = np.array(
    [VTK_QUAD],
    dtype=np.uint8,
)

vtk_cell_types = numpy_to_vtk(cell_types, deep=True)

offsets = np.array([0, 4], dtype=np.int64)

cells = CellArray(offsets=offsets, connectivity=connectivity_flat)

# ============================================================
# Create vtkUnstructuredGrid
# ============================================================

grid = UnstructuredGrid(points=points, cells=(vtk_cell_types, cells))


In [45]:
# -------------------------
# VTK rendering pipeline
# -------------------------

mapper = vtkDataSetMapper(input_data=grid)

actor = vtkActor(mapper=mapper)
#actor.property.representation = VTK_WIREFRAME
actor.property.line_width = 3

renderer = vtkRenderer()
renderer.AddActor(actor)
renderer.ResetCamera()


render_window = vtkRenderWindow()
render_window.AddRenderer(renderer)


In [46]:

# -------------------------
# Trame
# -------------------------

server = get_server()

with VAppLayout(server) as layout:
    #layout.title.set_text("VTK 4-Node Quad")
    view = vtk_widgets.VtkLocalView(render_window)

await layout.display_cell(height="600px")